# Redrob Hackathon — Kaggle Pipeline (2x T4 GPU)

**Runtime:** GPU T4 x2 (Settings → Accelerator → GPU T4 x2)

**Steps:**
1. Setup — clone repo, install deps
2. Upload candidates.jsonl as Kaggle dataset
3. Precompute embeddings (dual GPU, ~15 min)
4. Rank → `jashwanth_s.csv`
5. Validate & download

## Step 1 — Setup

In [ ]:
# Verify GPUs
import subprocess, torch
result = subprocess.run(['nvidia-smi', '--list-gpus'], capture_output=True, text=True)
print(result.stdout)
print(f"PyTorch sees {torch.cuda.device_count()} GPU(s)")
assert torch.cuda.device_count() >= 1, "No GPU detected! Enable GPU T4 x2 in Settings."

In [ ]:
# Clone repo
!git clone https://github.com/JASHWANTHS07/india-runs.git
%cd india-runs

In [ ]:
# Install dependencies (torch is pre-installed on Kaggle)
!pip install -q sentence-transformers tqdm pandas pyarrow
print("Dependencies installed.")

## Step 2 — Link Data

Add `candidates.jsonl` as a Kaggle dataset:
1. Go to kaggle.com/datasets → New Dataset → upload `candidates.jsonl`
2. In this notebook: Add Data (right panel) → search your dataset → Add
3. Update the path below to match your dataset location.

In [ ]:
# Set the path to your candidates.jsonl
# Default Kaggle dataset mount pattern: /kaggle/input/<dataset-name>/candidates.jsonl
# Update this path to match YOUR dataset name:
import os, glob

# Auto-detect candidates.jsonl in /kaggle/input/
found = glob.glob('/kaggle/input/**/candidates.jsonl', recursive=True)
if found:
    CANDIDATES_PATH = found[0]
    print(f"Auto-detected: {CANDIDATES_PATH}")
else:
    # Fallback: set manually
    CANDIDATES_PATH = '/kaggle/input/your-dataset-name/candidates.jsonl'
    print(f"Not found automatically. Set CANDIDATES_PATH manually.")

size_mb = os.path.getsize(CANDIDATES_PATH) / 1e6
print(f"File size: {size_mb:.1f} MB")
assert size_mb > 400, "File looks too small — check the path"

## Step 3 — Precompute Embeddings (Dual GPU)

This custom cell uses both T4 GPUs via `DataParallel` for ~2x speedup.

**Expected time:** ~12-18 min with 2x T4.

In [ ]:
import sys, time, json, math
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

sys.path.insert(0, '.')
from src.features import extract_features

# --- Config ---
ARTIFACTS_DIR = Path('./artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)
MODEL_NAME = 'BAAI/bge-large-en-v1.5'
BATCH_SIZE = 256  # per-GPU batch size
NUM_GPUS = torch.cuda.device_count()
EFFECTIVE_BATCH = BATCH_SIZE * max(1, NUM_GPUS)

print(f"Using {NUM_GPUS} GPU(s), effective batch size: {EFFECTIVE_BATCH}")

# --- Load model ---
print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)

# Use DataParallel if 2 GPUs available
if NUM_GPUS > 1:
    print("Enabling DataParallel across GPUs...")
    # SentenceTransformer has built-in multi-GPU support via start_multi_process_pool
    pool = model.start_multi_process_pool()
    USE_POOL = True
else:
    model = model.to('cuda')
    USE_POOL = False

# --- Embed JD ---
JD_QUERY = (
    "Senior AI Engineer specializing in candidate ranking, retrieval systems, "
    "recommendation engines, semantic search, embeddings, vector databases, "
    "learning-to-rank, NLP, information retrieval, hybrid search, "
    "Python, PyTorch, TensorFlow, FAISS, Pinecone, Weaviate, Elasticsearch, "
    "production ML deployment, A/B testing, 5-9 years experience, India"
)
print("Embedding JD query...")
jd_emb = model.encode([JD_QUERY], normalize_embeddings=True, show_progress_bar=False)
jd_embedding = jd_emb[0].astype(np.float32)
np.save(ARTIFACTS_DIR / 'jd_embedding.npy', jd_embedding)
print(f"JD embedding saved: shape {jd_embedding.shape}")

# --- Process candidates ---
print(f"\nProcessing candidates from {CANDIDATES_PATH}...")
t0 = time.time()

all_features = []
all_texts = []
candidate_count = 0

with open(CANDIDATES_PATH, 'r') as f:
    for line in tqdm(f, desc="Extracting features"):
        candidate = json.loads(line)
        feat = extract_features(candidate)
        all_features.append(feat)
        all_texts.append(feat.profile_text)
        candidate_count += 1

feat_time = time.time() - t0
print(f"Feature extraction: {candidate_count} candidates in {feat_time:.1f}s")

# --- Batch embed all candidates ---
print(f"\nEmbedding {candidate_count} candidates...")
t1 = time.time()

if USE_POOL:
    # Multi-GPU encoding via process pool
    all_embeddings = model.encode_multi_process(
        all_texts,
        pool,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
    )
    model.stop_multi_process_pool(pool)
else:
    all_embeddings = model.encode(
        all_texts,
        batch_size=EFFECTIVE_BATCH,
        normalize_embeddings=True,
        show_progress_bar=True,
        device='cuda',
    )

embed_time = time.time() - t1
all_embeddings = np.array(all_embeddings, dtype=np.float32)
print(f"Embedding done: {all_embeddings.shape} in {embed_time:.1f}s")

# --- Save artifacts ---
np.save(ARTIFACTS_DIR / 'embeddings.npy', all_embeddings)

# Build features dataframe (drop profile_text — it's huge and only needed for embedding)
from dataclasses import asdict
rows = [asdict(feat) for feat in all_features]
df = pd.DataFrame(rows)
df.drop(columns=['profile_text'], inplace=True, errors='ignore')
df.to_parquet(ARTIFACTS_DIR / 'features.parquet', index=False)

total_time = time.time() - t0
print(f"\nAll artifacts saved to {ARTIFACTS_DIR}/")
print(f"  embeddings.npy  : {all_embeddings.shape} ({all_embeddings.nbytes/1e6:.0f} MB)")
print(f"  jd_embedding.npy: {jd_embedding.shape}")
print(f"  features.parquet: {len(df)} rows, {len(df.columns)} cols")
print(f"Total time: {total_time:.1f}s ({total_time/60:.1f} min)")

## Step 3b — Verify Artifacts

In [ ]:
import numpy as np
import pandas as pd

emb = np.load('artifacts/embeddings.npy')
jd  = np.load('artifacts/jd_embedding.npy')
df  = pd.read_parquet('artifacts/features.parquet')

print(f"embeddings.npy : {emb.shape}  ({emb.nbytes/1e6:.0f} MB)")
print(f"jd_embedding   : {jd.shape}")
print(f"features.parquet: {len(df)} rows, {len(df.columns)} cols")

assert emb.shape[1] == 1024, f"Unexpected embedding dim: {emb.shape[1]}"
assert jd.shape == (1024,), f"Unexpected JD shape: {jd.shape}"
assert len(df) == emb.shape[0], f"Row mismatch: {len(df)} vs {emb.shape[0]}"
print("Artifacts OK")

## Step 4 — Rank Candidates (CPU)

**Expected time:** < 2 min.

In [ ]:
!python src/rank.py     --artifacts ./artifacts     --out ./jashwanth_s.csv

## Step 5 — Validate & Download

In [ ]:
import pandas as pd

df = pd.read_csv('jashwanth_s.csv')

assert len(df) == 100, f"Expected 100 rows, got {len(df)}"
assert set(df['rank']) == set(range(1, 101)), "Ranks must be 1-100"
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1)), "Scores must be non-increasing"
assert df['candidate_id'].str.match(r'^CAND_\d{7}$').all(), "Bad candidate_id format"

# Honeypot check
import sys
sys.path.insert(0, '.')
from src.features import CandidateFeatures
from src.honeypot import is_honeypot
from dataclasses import fields

feat_df = pd.read_parquet('artifacts/features.parquet')
top100_ids = set(df['candidate_id'])
top100_feats = feat_df[feat_df['candidate_id'].isin(top100_ids)]

honeypots = []
for _, row in top100_feats.iterrows():
    kwargs = {f.name: row[f.name] for f in fields(CandidateFeatures) if f.name in row.index}
    kwargs.setdefault('profile_text', '')
    if is_honeypot(CandidateFeatures(**kwargs)):
        honeypots.append(row['candidate_id'])

print(f"Rows         : {len(df)}")
print(f"Score range  : {scores[0]:.4f} -> {scores[-1]:.4f}")
print(f"Honeypots    : {len(honeypots)} (must be 0)")
assert len(honeypots) == 0, f"Honeypots in top 100: {honeypots}"
print("All checks passed.")

In [ ]:
# Inspect top 10
pd.set_option('display.max_colwidth', 100)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# Inspect reasoning diversity (Stage 4 check)
reasonings = df['reasoning'].tolist()
unique_starts = set(r[:50] for r in reasonings)
print(f"Unique reasoning starts (first 50 chars): {len(unique_starts)}/100")
assert len(unique_starts) >= 80, "Too many identical reasonings — Stage 4 will penalize"
print("Reasoning diversity OK")

In [ ]:
# Copy to /kaggle/working/ for download
import shutil
shutil.copy('jashwanth_s.csv', '/kaggle/working/jashwanth_s.csv')
print("jashwanth_s.csv copied to /kaggle/working/ — use 'Save Version' to download.")
print("Go to: Output tab → jashwanth_s.csv → Download")